# 06 · DOSE at ADM1: Gate D1

Fork `fork/corridors-dose`, Part 2. Rules: [`docs/successors/dose-prespec.md`](../../docs/successors/dose-prespec.md)
§2; outcomes recorded in §4. Loads neither lights nor temperature.

◇ D1: at least 150 DOSE regions with output in ≥ 15 of 2002–2020 **and** an active-mine count
that changes within 2002–2020, in at least 15 countries.

In [ ]:
import sys
from pathlib import Path
import duckdb

PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())

In [ ]:
DATA = PROJECT / 'data_nobackup'
con = duckdb.connect()
con.execute(f"ATTACH '{DATA}/prepared/snl_mining/misc/snl_mining_prepared.duckdb' AS s (READ_ONLY)")
D = f'{DATA}/prepared/dose/adm/dose_gid1.parquet'
con.execute(f"""
CREATE TABLE cov AS
SELECT GID_1_gadm41 AS g, any_value(GID_0) AS GID_0,
       count(*) FILTER (WHERE year BETWEEN 2002 AND 2020 AND grp_pc_lcu_2015 IS NOT NULL AND pop IS NOT NULL) AS n_years
FROM '{D}' GROUP BY 1""")
con.execute("""
CREATE TABLE mines AS
SELECT r.g, max(coalesce(c.mines, 0)) - min(coalesce(c.mines, 0)) AS rng, max(coalesce(c.mines, 0)) AS mx
FROM cov r CROSS JOIN range(2002, 2021) t(yr)
LEFT JOIN (SELECT adm_code, year, value AS mines FROM s.adm1_year_counts) c ON c.adm_code = r.g AND c.year = t.yr
GROUP BY r.g""")
res = con.execute("""
SELECT count(*) FILTER (WHERE n_years >= 15) AS covered,
       count(*) FILTER (WHERE n_years >= 15 AND mx > 0) AS covered_with_mine,
       count(*) FILTER (WHERE n_years >= 15 AND rng > 0) AS exposed,
       count(DISTINCT GID_0) FILTER (WHERE n_years >= 15 AND rng > 0) AS exposed_countries
FROM cov JOIN mines USING (g)""").df()
ok = res.exposed[0] >= 150 and res.exposed_countries[0] >= 15
print(f"◇ D1: {res.exposed[0]} exposed regions in {res.exposed_countries[0]} countries -> {'pass' if ok else 'FAIL'}")
display(res)
display(con.execute("""SELECT GID_0, count(*) AS exposed_regions FROM cov JOIN mines USING (g)
                     WHERE n_years >= 15 AND rng > 0 GROUP BY 1 ORDER BY 2 DESC""").df())